In [ ]:
pip install pillow

In [ ]:
import tkinter as tk
import random
import math

WIDTH = 800
HEIGHT = 600
GRAVITY = 0.3
FRICTION = 0.99
NUM_BALLS = 50

class Ball:
    def __init__(self, canvas, x, y, radius, color):
        self.canvas = canvas
        self.radius = radius
        self.id = canvas.create_oval(x - radius, y - radius, x + radius, y + radius, fill=color)
        self.vx = random.uniform(-2, 2)
        self.vy = random.uniform(-2, 2)
    
    def move(self):
        self.vy += GRAVITY
        self.vx *= FRICTION
        self.vy *= FRICTION
        coords = self.canvas.coords(self.id)
        x1, y1, x2, y2 = coords
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        # 벽 충돌
        if x1 <= 0 and self.vx < 0 or x2 >= WIDTH and self.vx > 0:
            self.vx = -self.vx
        if y1 <= 0 and self.vy < 0 or y2 >= HEIGHT and self.vy > 0:
            self.vy = -self.vy

        self.canvas.move(self.id, self.vx, self.vy)

    def get_position(self):
        x1, y1, x2, y2 = self.canvas.coords(self.id)
        return ((x1 + x2) / 2, (y1 + y2) / 2)

    def set_velocity(self, vx, vy):
        self.vx, self.vy = vx, vy


def check_collision(ball1, ball2):
    x1, y1 = ball1.get_position()
    x2, y2 = ball2.get_position()
    dx = x2 - x1
    dy = y2 - y1
    dist = math.hypot(dx, dy)
    min_dist = ball1.radius + ball2.radius

    if dist < min_dist and dist > 0:
        angle = math.atan2(dy, dx)

        vx1, vy1 = ball1.vx, ball1.vy
        vx2, vy2 = ball2.vx, ball2.vy

        ball1.set_velocity(vx2, vy2)
        ball2.set_velocity(vx1, vy1)

        overlap = 0.5 * (min_dist - dist + 1)
        ball1.canvas.move(ball1.id, -math.cos(angle) * overlap, -math.sin(angle) * overlap)
        ball2.canvas.move(ball2.id, math.cos(angle) * overlap, math.sin(angle) * overlap)

def update():
    for ball in balls:
        ball.move()

    for i in range(NUM_BALLS):
        for j in range(i + 1, NUM_BALLS):
            check_collision(balls[i], balls[j])

    root.after(16, update)

root = tk.Tk()
root.title("Ball Collision Simulation - 50 Balls")
canvas = tk.Canvas(root, width=WIDTH, height=HEIGHT, bg="white")
canvas.pack()

balls = []
colors = ["red", "blue", "green", "orange", "purple", "cyan", "magenta", "yellow", "black", "gray"]
center_x = WIDTH // 2
center_y = HEIGHT // 2

# 격자 배치
cols = int(NUM_BALLS**0.5)
spacing = 40
for i in range(NUM_BALLS):
    row = i // cols
    col = i % cols
    radius = random.randint(10, 20)
    x = center_x + (col - cols // 2) * spacing
    y = center_y + (row - cols // 2) * spacing
    color = random.choice(colors)
    ball = Ball(canvas, x, y, radius, color)
    balls.append(ball)

update()
root.mainloop()

In [ ]:
# simulation_recorder.py
import random, math, time, json, os
from PIL import Image
from tqdm import tqdm

def run_recorder():
    # 결정론적 시드
    random.seed(1234)
    # 설정
    WIDTH, HEIGHT = 1200, 800
    NUM_BALLS = 600
    DT = 0.01
    STEPS = 700
    SAVE_FILE = "simulation_record.json"

    # 이미지 선택 (콘솔 입력)
    image_path = input("이미지 파일 경로를 입력하세요: ")
    if not os.path.exists(image_path):
        print("이미지 파일이 없습니다.")
        return
    image = Image.open(image_path).resize((WIDTH, HEIGHT)).convert('RGB')

    # 초기 상태 및 스폰 스케줄 생성
    initial_states = []  # 각 공의 r, vx, vy
    spawn_steps = []     # 각 공의 스폰 프레임
    for i in range(NUM_BALLS):
        r = random.randint(5, 15)
        angle = random.uniform(0, 2*math.pi)
        speed = random.uniform(60, 120)
        vx = speed * math.cos(angle)
        vy = speed * math.sin(angle)
        initial_states.append({'r': r, 'vx': vx, 'vy': vy})
        # 균등 분포된 스폰 타이밍
        spawn_steps.append(int(i * STEPS / NUM_BALLS))

    # 시뮬레이션 준비
    trajectories = []
    color_map = {}
    active_balls = {}  # index -> Ball instance

    # Ball 정의
    class Ball:
        def __init__(self, idx, r, vx, vy):
            self.idx = idx
            self.r = r
            self.mass = r * r
            self.mass_inv = 1 # / self.mass
            self.x = WIDTH / 2
            self.y = HEIGHT / 2
            self.vx = vx
            self.vy = vy
        def step(self):
            self.x += self.vx * DT
            self.y += self.vy * DT
            # 경계 반사
            if self.x - self.r < 0:
                self.x = self.r; self.vx = abs(self.vx)
            if self.x + self.r > WIDTH:
                self.x = WIDTH - self.r; self.vx = -abs(self.vx)
            if self.y - self.r < 0:
                self.y = self.r; self.vy = abs(self.vy)
            if self.y + self.r > HEIGHT:
                self.y = HEIGHT - self.r; self.vy = -abs(self.vy)

    def resolve_collision(b1, b2):
        dx = b2.x - b1.x; dy = b2.y - b1.y
        dist = math.hypot(dx, dy)
        min_dist = b1.r + b2.r
        if dist <= 0 or dist >= min_dist:
            return
        nx, ny = dx/dist, dy/dist
        overlap = 0.5 * (min_dist - dist)
        b1.x -= nx * overlap; b1.y -= ny * overlap
        b2.x += nx * overlap; b2.y += ny * overlap
        dvx = b1.vx - b2.vx; dvy = b1.vy - b2.vy
        rel_vel = dvx * nx + dvy * ny
        if rel_vel > 0:
            return
        j = -(2 * rel_vel) / (b1.mass_inv + b2.mass_inv)
        b1.vx += j * nx * b1.mass_inv; b1.vy += j * ny * b1.mass_inv
        b2.vx -= j * nx * b2.mass_inv; b2.vy -= j * ny * b2.mass_inv

    # 시뮬레이션 실행
    t0 = time.time()
    for step in tqdm(range(STEPS)):
        # 스폰 단계
        for idx, spawn_step in enumerate(spawn_steps):
            if spawn_step == step:
                st = initial_states[idx]
                active_balls[idx] = Ball(idx, st['r'], st['vx'], st['vy'])
        # 이동 단계
        for b in active_balls.values():
            b.step()
        # 충돌 처리
        indices = list(active_balls.keys())
        for i in range(len(indices)):
            for j in range(i+1, len(indices)):
                resolve_collision(active_balls[indices[i]], active_balls[indices[j]])
        # 기록 프레임 구축
        frame = []
        for idx, st in enumerate(initial_states):
            if idx in active_balls:
                b = active_balls[idx]
                frame.append((b.x, b.y))
            else:
                # 아직 스폰되지 않은 공: 화면 밖 위치
                frame.append((-st['r'], -st['r']))
        trajectories.append(frame)

    # 색상 매핑 (최종 프레임)
    last = trajectories[-1]
    for idx, (x, y) in enumerate(last):
        px = min(max(int(x), 0), WIDTH-1)
        py = min(max(int(y), 0), HEIGHT-1)
        r0, g0, b0 = image.getpixel((px, py))
        color_map[str(idx)] = f"#{r0:02x}{g0:02x}{b0:02x}"

    total_time = time.time() - t0
    # 기록 저장
    with open(SAVE_FILE, 'w') as f:
        json.dump({
            'image_path': image_path,
            'initial_states': initial_states,
            'trajectories': trajectories,
            'color_map': color_map,
            'total_time': total_time,
            'dt': DT
        }, f)

    print(f"Recorded {NUM_BALLS} balls with sequential spawn over {STEPS} steps in {total_time:.2f}s to {SAVE_FILE}")

if __name__ == '__main__':
    run_recorder()

In [10]:
# simulation_player.py
import tkinter as tk
import time, json, os

SAVE_FILE = "simulation_record.json"
if not os.path.exists(SAVE_FILE):
    raise SystemExit("기록 파일을 찾을 수 없습니다.")
with open(SAVE_FILE, 'r') as f:
    data = json.load(f)

WIDTH, HEIGHT = 1200, 800
initial_states = data['initial_states']
trajectories = data['trajectories']
color_map = data['color_map']
DT = 0.005 # data.get('dt', 0.03)

root = tk.Tk()
root.title("Simulation Player")
canvas = tk.Canvas(root, width=WIDTH, height=HEIGHT, bg='white')
canvas.pack()
balls = []

# 초기 배치
first_frame = trajectories[0]
for idx, st in enumerate(initial_states):
    r = st['r']
    x, y = first_frame[idx]
    col = color_map.get(str(idx), "#888888")
    ball_id = canvas.create_oval(x-r, y-r, x+r, y+r, fill=col, outline="")
    balls.append(ball_id)

frame_index = 0

def play():
    global frame_index
    if frame_index >= len(trajectories):
        return
    for idx, (x, y) in enumerate(trajectories[frame_index]):
        r = initial_states[idx]['r']
        canvas.coords(balls[idx], x-r, y-r, x+r, y+r)
    frame_index += 1
    root.after(int(DT * 1000), play)

# 재생 시작
root.after(0, play)
root.mainloop()
